In [1]:
from transformers import pipeline
import torch
import transformers

In [5]:
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer and model
model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
gpt2 = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [6]:
from transformers import pipeline
import torch
pipe = pipeline("text-generation", model="openai-community/gpt2")
prompt ="what is transformer in a LLM  "
output=pipe(prompt)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [7]:
print(output[0]["generated_text"])

what is transformer in a LLM   (not a transformer in a transformer in an LLM  (not a transformer in a transformer in an LLM  (not a transformer in a transformer in an LLM  (not a transformer in a transformer in an LLM  (not a transformer in a transformer in a transformer in an LLM  (not a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer in a transformer i

In [8]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("gpt2")

In [9]:
sentence="i learn machine learning to enhance workplace"
token_ids=tokenizer(sentence,return_tensors="pt").input_ids
outputs=gpt2(token_ids).logits[0,-1]
final_logits=torch.topk(outputs,10)
final_logits
for index in final_logits.indices:
  print(tokenizer.decode(index))

 productivity
 safety
 performance
 and
 communication
 learning
 interactions
 security
 skills
 experience


In [10]:
# greedy decode sampling strategy
def greedy_decode(logits):
    """return token index with the highest probability"""
    return torch.argmax(logits,dim=-1)

tokenizer.decode(greedy_decode(outputs))

def top_k_sampling(logits,k=50):
    """keeps only top k logits, normalize them into probabilitiy.
    then sample one token form the filtered distribution.
    """
    values,indices=torch.topk(logits,k)
    probs=F.softmax(values,dim=-1)
    sampled=torch.multinomial(probs,1)
    return indices[sampled]

def top_p_sampling(logits,p=0.9):
    """
    sort tokens by probability, keep smallest number whoese culumative probability exceed p, then sample one token .
    """
    sorted_logits,sorted_indices=torch.sort(logits,descending=True)
    sorted_probs=F.softmax(sorted_logits,dim=-1)
    cumulative_probs=sorted_probs.cumsum(dim=-1)
    
    # mask token outside nuclues
    mask=cumulative_probs>p
    sorted_logits[mask]=float("-inf")
    
    # sample from the filtered logits
    filtered_probs=F.softmax(sorted_logits,dim=-1)
    sampled=torch.multinomial(filtered_probs,1)
    
    # return the token index in original vocabulary
    return sorted_indices[sampled]

    

In [11]:
top_k_sampling(outputs)
tokenizer.decode(top_k_sampling(outputs))

' security'

In [12]:
top_p_sampling(outputs,p=0.9)
tokenizer.decode(top_p_sampling(outputs,p=0.9))

' learning'

In [13]:
# temperature sampling

def temperature_sampling(logits,temperature=.5):
    """scale logits by temperature before sampling.
    lower temperature -> sharper distribution -> more deterministic approach."""
    scaled=logits/temperature
    probs=F.softmax(scaled,dim=-1)
    return torch.multinomial(probs,1)

In [10]:
tokenizer.decode(temperature_sampling(outputs,temperature=1))

' lives'

In [11]:
# randlom smapling

def random_sampling(logits):
    """
    sample direcely from softmax distribution without filtering or scaling.`
    """
    probs=F.softmax(logits,dim=-1)
    return torch.multinomial(probs,1)

random_sampling(outputs)
tokenizer.decode(random_sampling(outputs))


' productivity'

In [12]:
# all sampling 
sentence= "today i decided to go to the college and play vollyball with my"
inputs=tokenizer(sentence,return_tensors="pt")
output=gpt2(**inputs)
logits=output.logits[0,-1]

print("Greedy decode:",tokenizer.decode([greedy_decode(logits)])) 
print("Top-k decode:",tokenizer.decode(top_k_sampling(logits,k=10)))
print("Top-p decode:",tokenizer.decode(top_p_sampling(logits,p=0.9)))
print("Temperature decode:",tokenizer.decode(temperature_sampling(logits,temperature=0.7)))
print("Random decode:",tokenizer.decode(random_sampling(logits)))

Greedy decode:  friends
Top-k decode:  friends
Top-p decode:  mom
Temperature decode:  dad
Random decode:  friends


In [21]:
sentence="i learn machine learning to enhance our understanding"
token_id=tokenizer(sentence,return_tensors="pt").input_ids
outputs=gpt2(token_id).logits
outputs=torch.softmax(outputs[0,-1],dim=-1)

top10=torch.topk(outputs,10)
for index, value in zip(top10.indices, top10.values):
    print(f"{tokenizer.decode(index)} -- {value:.1%}")

 of -- 90.6%
 and -- 3.5%
. -- 1.4%
, -- 1.0%
 about -- 0.6%
 in -- 0.3%
 for -- 0.2%
 on -- 0.2%
 by -- 0.2%
 as -- 0.1%


# Sentiment Analysis


In [23]:
from datasets import load_dataset
ds=load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

c:\Users\vikas\OneDrive\Documents\hugging-face\hugging-face\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vikas\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [24]:
type(ds)

datasets.dataset_dict.DatasetDict

In [25]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [28]:
ds["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [29]:
ds["train"].to_pandas()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [30]:
my_dataset_df=ds["train"].to_pandas()

In [31]:
len(my_dataset_df["text"])

25000

In [11]:
from transformers import pipeline
classifier=pipeline("sentiment-analysis") # default model is distilbert-base-uncased-finetuned-sst-2-english

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [34]:
classifier("I love this movie! It's fantastic.")

[{'label': 'POSITIVE', 'score': 0.9998773336410522}]

In [35]:
classifier("i feel bored and fatigued after watching this movie.")

[{'label': 'NEGATIVE', 'score': 0.9997413754463196}]

In [43]:
classifier("i feel amazing in the initial part of the movie but as the story progresses, it become quite predictable and i kept getting bored. but the cinematography is stunning. acting is also good, but the story was pretty lame overall ")

[{'label': 'NEGATIVE', 'score': 0.7982100248336792}]

In [49]:
def score(review_text):
    return classifier(review_text[:50])[0]["label"]


In [47]:
my_dataset_df["model_prediction"]=my_dataset_df["text"].apply(score)

In [50]:
my_dataset_df[["label","model_prediction"]][:20]

,label,model_prediction
0,0,NEGATIVE
1,0,NEGATIVE
2,0,NEGATIVE
3,0,NEGATIVE
4,0,NEGATIVE
5,0,POSITIVE
6,0,NEGATIVE
7,0,NEGATIVE
8,0,NEGATIVE
9,0,POSITIVE


In [ ]:
review=my_dataset_df.iloc[0]["text"]
 

'POSITIVE'

# finbert 
### financial version of bert model for sentiment analysis

In [48]:
finbert=pipeline("sentiment-analysis",model="ProsusAI/finbert")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

c:\Users\vikas\OneDrive\Documents\hugging-face\hugging-face\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vikas\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [55]:
sentence="The company's revenue exceeded expectations, leading to a positive outlook for the next quarter."
finbert(sentence)[0]["score"],finbert(sentence)[0]["label"]

(0.9561997056007385, 'positive')

In [56]:
sentence="the company's revenue fell short of expectations, resulting in a negative outlook for the next quarter."
finbert(sentence)[0]["label"]

'negative'

In [67]:
sentence_list=[
    "The company's revenue exceeded expectations, leading to a positive outlook for the next quarter.",
    "the company's revenue fell short of expectations, resulting in a negative outlook for the next quarter.",
    "the ipo of the company was a huge success, with shares soaring on the first day of trading.",
    "the ipo of the company was a disaster, with shares plummeting on the first day of trading.",]
finbert(sentence_list)

[{'label': 'positive', 'score': 0.9561997056007385},
 {'label': 'negative', 'score': 0.9742463827133179},
 {'label': 'positive', 'score': 0.9196693897247314},
 {'label': 'negative', 'score': 0.9668915271759033}]

### Named Entity Recognition is a subtask of information extraction that seeks to locate and classify named entities mentioned in unstructured text into pre-defined categories such as the names of persons, organizations, locations, expressions of times, quantities, monetary values, percentages, etc.

In [6]:
sentence="Apple will announce its new iphone 18 model in september 2026 with several new features and improvements over the previous model."

ner=pipeline("ner")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\vikas\OneDrive\Documents\hugging-face\hugging-face\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vikas\.cache\huggingface\hub\models--dbmdz--bert-large-cased-finetuned-conll03-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

In [7]:
sentence

'Apple will announce its new iphone 18 model in september 2026 with several new features and improvements over the previous model.'

In [8]:
ner(sentence)

[{'entity': 'I-ORG',
  'score': np.float32(0.99673957),
  'index': 1,
  'word': 'Apple',
  'start': 0,
  'end': 5},
 {'entity': 'I-MISC',
  'score': np.float32(0.8967284),
  'index': 6,
  'word': 'i',
  'start': 28,
  'end': 29}]

### QnA

In [2]:
qa = pipeline(
    "question-answering"
)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

c:\Users\vikas\OneDrive\Documents\hugging-face\hugging-face\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vikas\.cache\huggingface\hub\models--distilbert--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet'

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [3]:
context="""
FinBERT is a pre-trained Natural Language Processing (NLP) model specifically designed for financial sentiment analysis.  Built by ProsusAI, it is based on Google’s BERT architecture and is further trained on a large financial corpus to understand the specialized language and context of finance, such as earnings reports, news articles, and financial statements. 

The model classifies text into three sentiment categories: Positive, Negative, and Neutral, providing softmax probability outputs for each class.  It is widely recognized for outperforming generic sentiment models and traditional machine learning methods, achieving high accuracy (e.g., 89 percent on the Financial PhraseBank dataset) by leveraging domain-specific training data rather than general-purpose corpora
"""

question="what is finbert model and what is its use case"

In [4]:
qa(question=question,context=context)

{'score': 0.07724271714687347,
 'start': 14,
 'end': 53,
 'answer': 'pre-trained Natural Language Processing'}

In [5]:
question ="finbert model is based on which architecture and what is its accuracy"
qa(question=question,context=context)

{'score': 0.4476797878742218,
 'start': 157,
 'end': 170,
 'answer': 'Google’s BERT'}

In [7]:
question ="what is its accuracy"
qa(question=question,context=context)

{'score': 0.3072962164878845,
 'start': 633,
 'end': 646,
 'answer': 'high accuracy'}

## machine translation

In [8]:
translate=pipeline("translation_en_to_fr") # english to french translation

No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

c:\Users\vikas\OneDrive\Documents\hugging-face\hugging-face\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vikas\.cache\huggingface\hub\models--google-t5--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fall

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [21]:
sentence="what is your occupation ?"
translate(sentence)[0]["translation_text"]

'Quelle est votre profession ?'